In [1]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

import numpy as np
from sklearn.metrics import f1_score

In [2]:
train_df = pd.read_parquet("train.parquet")
val_df = pd.read_parquet("validation.parquet")


train_df_ko = train_df[train_df["lang"] == "ko"]
train_df_ar = train_df[train_df["lang"] == "ar"]
train_df_te = train_df[train_df["lang"] == "te"]
val_df_ko = val_df[val_df["lang"] == "ko"]
val_df_ar = val_df[val_df["lang"] == "ar"]
val_df_te = val_df[val_df["lang"] == "te"]

print(len(train_df_ko))
train_df_ko

2422


,question,context,lang,answerable,answer_start,answer,answer_inlang
4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None
4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,None
4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,None
4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),None
4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,None
...,...,...,...,...,...,...,...
15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,None
15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,None
15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,None
15341,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,Munich (; ; ) is the capital and most populous...,ko,True,205,Berlin,None


In [3]:
train_dataset_ko = Dataset.from_pandas(train_df_ko)
val_dataset_ko = Dataset.from_pandas(val_df_ko)
train_dataset_ar = Dataset.from_pandas(train_df_ar)
val_dataset_ar = Dataset.from_pandas(val_df_ar)
train_dataset_te = Dataset.from_pandas(train_df_te)
val_dataset_te = Dataset.from_pandas(val_df_te)

In [ ]:
def preprocess_function(examples):

    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=512,
        padding='max_length',
        return_offsets_mapping=True
    )

    labels_batch = []

    for i in range(len(examples['question'])):
        labels = [-100] * len(tokenized['input_ids'][i])

        if examples['answerable'][i] and examples['answer_start'][i] is not None:
            answer_start = int(examples['answer_start'][i])
            answer_end = answer_start + len(examples['answer'][i])


            offset_mapping = tokenized['offset_mapping'][i]
            first_token = True

            for idx, (start, end) in enumerate(offset_mapping):
                if start == 0 and end == 0:  
                    continue


                if start >= answer_start and end <= answer_end:
                    if first_token:
                        labels[idx] = 1 
                        first_token = False
                    else:
                        labels[idx] = 2 
                elif start < answer_end and end > answer_start: 
                    if first_token:
                        labels[idx] = 1
                        first_token = False
                    else:
                        labels[idx] = 2
                elif end > answer_start:
                    break

        labels_batch.append(labels)

    tokenized['labels'] = labels_batch
    return tokenized


In [5]:
model_name = "microsoft/infoxlm-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label={0: 'O', 1: 'B-ANS', 2: 'I-ANS'},
    label2id={'O': 0, 'B-ANS': 1, 'I-ANS': 2}
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at microsoft/infoxlm-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
train_ko = train_dataset_ko.map(preprocess_function, batched=True)
train_ar = train_dataset_ar.map(preprocess_function, batched=True)
train_te = train_dataset_te.map(preprocess_function, batched=True)

Map:   0%|          | 0/2422 [00:00<?, ? examples/s]

Map:   0%|          | 0/2558 [00:00<?, ? examples/s]

Map:   0%|          | 0/1355 [00:00<?, ? examples/s]

In [ ]:



def preprocess_val(dataset):
    dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names
    )

    dataset = dataset.filter(lambda x: any(l != -100 for l in x["labels"]))
    return dataset

val_ar = preprocess_val(val_dataset_ar)
val_ko = preprocess_val(val_dataset_ko)
val_te = preprocess_val(val_dataset_te)



Map:   0%|          | 0/415 [00:00<?, ? examples/s]

Filter:   0%|          | 0/415 [00:00<?, ? examples/s]

Map:   0%|          | 0/356 [00:00<?, ? examples/s]

Filter:   0%|          | 0/356 [00:00<?, ? examples/s]

Map:   0%|          | 0/384 [00:00<?, ? examples/s]

Filter:   0%|          | 0/384 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    eval_strategy='steps',
    eval_steps=250,
    load_best_model_at_end=True,
    report_to='none'
)

trainer_ko = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ko,
    eval_dataset=val_ko,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)

trainer_ar = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ar,
    eval_dataset=val_ar,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)

trainer_te = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_te,
    eval_dataset=val_te,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)


In [9]:

def compute_token_f1(eval_dataset, trainer):
    predictions, labels, _ = trainer.predict(eval_dataset)
    preds = np.argmax(predictions, axis=-1)

    true_labels, true_preds = [], []

    for l_seq, p_seq in zip(labels, preds):
        for l, p in zip(l_seq, p_seq):
            if l != -100:
                true_labels.append(l)
                true_preds.append(p)

    f1 = f1_score(true_labels, true_preds, average="macro")
    return f1



## Arabic (restart session to get new model and run only Arabic training)

In [11]:
trainer_ar.train()


Step,Training Loss,Validation Loss
250,No log,0.300460
500,0.377300,0.237810
750,0.377300,0.217793
1000,0.244700,0.222320
1250,0.244700,0.217894
1500,0.189000,0.210334


TrainOutput(global_step=1600, training_loss=0.26470062375068665, metrics={'train_runtime': 1599.5007, 'train_samples_per_second': 7.996, 'train_steps_per_second': 1.0, 'total_flos': 3342015733340160.0, 'train_loss': 0.26470062375068665, 'epoch': 5.0})

In [12]:
f1_ar = compute_token_f1(val_ar, trainer_ar)
print(f"AR F1: {f1_ar}")

AR F1: 0.8644199583489809


## Korean (restart session to get new model and run only korean training)

In [11]:
trainer_ko.train()

Step,Training Loss,Validation Loss
250,No log,0.222664
500,0.343700,0.183223
750,0.343700,0.172206
1000,0.221100,0.169775
1250,0.221100,0.168140
1500,0.181600,0.166360


TrainOutput(global_step=1515, training_loss=0.24839728452978355, metrics={'train_runtime': 1830.4342, 'train_samples_per_second': 6.616, 'train_steps_per_second': 0.828, 'total_flos': 3164332332349440.0, 'train_loss': 0.24839728452978355, 'epoch': 5.0})

In [ ]:
f1_ko = compute_token_f1(val_ko, trainer_ko)
print(f"KO F1: {f1_ko}")

AR F1: 0.8930686526406414


## Telugu (restart session to get new model and run only Telugu training)

In [10]:
trainer_te.train()

Step,Training Loss,Validation Loss
250,No log,0.272018
500,0.384200,0.265433
750,0.384200,0.242058


TrainOutput(global_step=850, training_loss=0.32742504344267004, metrics={'train_runtime': 1071.4857, 'train_samples_per_second': 6.323, 'train_steps_per_second': 0.793, 'total_flos': 1770301531929600.0, 'train_loss': 0.32742504344267004, 'epoch': 5.0})

In [ ]:
f1_te = compute_token_f1(val_te, trainer_te)
print(f"TE F1: {f1_te}")

AR F1: 0.8383356427014387
